#  CWSI Stress Prediction Model

**Crop Water Stress Index Prediction with Percentage Output and Automatic Stress Classification**

This notebook implements a single regression model that predicts CWSI values as percentages and automatically classifies stress levels using conditional logic.

---

##  Table of Contents
1. [Setup and Imports](#setup)
2. [Data Loading and Exploration](#data-loading)
3. [Data Preprocessing](#preprocessing)
4. [Model Training](#training)
5. [Model Evaluation](#evaluation)
6. [Stress Classification Logic](#classification)
7. [Model Persistence](#persistence)
8. [Inference Example](#inference)

## 1. Setup and Imports <a name="setup"></a>

Import required libraries for data processing, machine learning, and visualization.

In [ ]:
# Core data science libraries
import pandas as pd
import numpy as np

# Machine learning libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Model persistence
import joblib
import warnings
warnings.filterwarnings('ignore')

# Configure plotting style
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
try:
    plt.style.use('seaborn-v0_8')
except:
    plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print(" All libraries imported successfully")

 All libraries imported successfully


## 2. Data Loading and Exploration <a name="data-loading"></a>

Load the dataset and perform initial exploratory data analysis.

In [ ]:
# Load the dataset
DATA_PATH = 'salsabil_dataset_2000.csv'
df = pd.read_csv(DATA_PATH)

print(f" Dataset loaded successfully")
print(f"   • Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   • Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"\n Column overview:")
print(df.dtypes.to_string())

# Display first few rows
df.head().transpose()

 Dataset loaded successfully
   • Shape: 2000 rows × 25 columns
   • Memory usage: 863.4 KB

 Column overview:
id                    int64
date                 object
region               object
latitude            float64
longitude           float64
soil_type            object
crop_type            object
month                 int64
year                  int64
lst_celsius         float64
ndvi                float64
savi                float64
evi                 float64
ta_celsius          float64
rh_percent          float64
wind_ms             float64
solar_wm2           float64
vpd_kpa             float64
et0_mm_day          float64
soil_moisture       float64
field_capacity      float64
wilting_point       float64
irrigation_event      int64
cwsi                float64
stress_label         object


,0,1,2,3,4
id,1,2,3,4,5
date,2021-10-15,2023-08-02,2022-08-02,2022-06-06,2022-10-14
region,Sfax,Nabeul,Kairouan,Béja,Gabès
latitude,34.74,36.45,35.67,36.73,33.88
longitude,10.76,10.73,10.1,9.18,9.79
soil_type,sandy-loam,sandy,clay-loam,loam,sandy
crop_type,Potato,Barley,Potato,Barley,Tomato
month,10,8,8,6,10
year,2021,2023,2022,2022,2022
lst_celsius,23.67,36.72,31.19,31.11,32.42


In [ ]:
# Dataset summary statistics
print(" Dataset Summary:")
print("=" * 50)
print(f"Total samples: {len(df)}")
print(f"CWSI range: {df['cwsi'].min():.3f} - {df['cwsi'].max():.3f}")
print(f"CWSI mean: {df['cwsi'].mean():.3f} ± {df['cwsi'].std():.3f}")

# Stress label distribution
print(f"\n Stress Level Distribution:")
stress_counts = df['stress_label'].value_counts().sort_index()
for label, count in stress_counts.items():
    percentage = (count / len(df)) * 100
    print(f"   {label:<8}: {count:>4} samples ({percentage:>5.1f}%)")

# Check for missing values
missing_values = df.isnull().sum()
if missing_values.sum() == 0:
    print("\n No missing values detected")
else:
    print(f"\n  Missing values found:")
    print(missing_values[missing_values > 0])

 Dataset Summary:
Total samples: 2000
CWSI range: 0.000 - 1.000
CWSI mean: 0.437 ± 0.252

 Stress Level Distribution:
   extreme :   97 samples (  4.9%)
   high    :  143 samples (  7.1%)
   low     :  500 samples ( 25.0%)
   medium  :  560 samples ( 28.0%)
   mild    :  700 samples ( 35.0%)

 No missing values detected


## 3. Data Preprocessing <a name="preprocessing"></a>

Prepare the data for model training by encoding categorical variables and selecting features.

In [ ]:
# Feature selection - exclude non-predictive columns
EXCLUDED_COLUMNS = ['id', 'date', 'year', 'stress_label', 'cwsi']  # cwsi is the target variable
FEATURE_COLUMNS = [col for col in df.columns if col not in EXCLUDED_COLUMNS]

print(f" Selected {len(FEATURE_COLUMNS)} features for training:")
for i, feature in enumerate(FEATURE_COLUMNS, 1):
    print(f"   {i:2d}. {feature}")

# Identify categorical columns for encoding
categorical_columns = ['region', 'soil_type', 'crop_type']
numerical_columns = [col for col in FEATURE_COLUMNS if col not in categorical_columns]

print(f"\n Data types:")
print(f"   • Categorical features: {len(categorical_columns)}")
print(f"   • Numerical features: {len(numerical_columns)}")
print(f"   • Target variable: cwsi")

 Selected 20 features for training:
    1. region
    2. latitude
    3. longitude
    4. soil_type
    5. crop_type
    6. month
    7. lst_celsius
    8. ndvi
    9. savi
   10. evi
   11. ta_celsius
   12. rh_percent
   13. wind_ms
   14. solar_wm2
   15. vpd_kpa
   16. et0_mm_day
   17. soil_moisture
   18. field_capacity
   19. wilting_point
   20. irrigation_event

 Data types:
   • Categorical features: 3
   • Numerical features: 17
   • Target variable: cwsi


In [ ]:
# Encode categorical variables
df_processed = df.copy()
label_encoders = {}

print(" Encoding categorical variables:")
for col in categorical_columns:
    encoder = LabelEncoder()
    df_processed[col] = encoder.fit_transform(df_processed[col])
    label_encoders[col] = encoder
    
    # Display encoding mapping
    mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
    print(f"   {col}: {mapping}")

# Prepare feature matrix and target vector
X = df_processed[FEATURE_COLUMNS].values
y = df_processed['cwsi'].values

print(f"\n Feature matrix shape: {X.shape}")
print(f" Target vector shape: {y.shape}")
print(f" Target range: {y.min():.3f} - {y.max():.3f}")

 Encoding categorical variables:
   region: {'Bizerte': np.int64(0), 'Béja': np.int64(1), 'Gabès': np.int64(2), 'Gafsa': np.int64(3), 'Kairouan': np.int64(4), 'Nabeul': np.int64(5), 'Sfax': np.int64(6), 'Sidi Bouzid': np.int64(7)}
   soil_type: {'clay-loam': np.int64(0), 'loam': np.int64(1), 'sandy': np.int64(2), 'sandy-loam': np.int64(3)}
   crop_type: {'Barley': np.int64(0), 'Citrus': np.int64(1), 'Olive': np.int64(2), 'Pepper': np.int64(3), 'Potato': np.int64(4), 'Tomato': np.int64(5), 'Wheat': np.int64(6)}

 Feature matrix shape: (2000, 20)
 Target vector shape: (2000,)
 Target range: 0.000 - 1.000


In [ ]:
# Train-test split
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f" Data split completed:")
print(f"   • Training set: {X_train.shape[0]} samples ({(1-TEST_SIZE)*100:.0f}%)")
print(f"   • Test set: {X_test.shape[0]} samples ({TEST_SIZE*100:.0f}%)")
print(f"   • Features per sample: {X_train.shape[1]}")

 Data split completed:
   • Training set: 1600 samples (80%)
   • Test set: 400 samples (20%)
   • Features per sample: 20


## 4. Model Training <a name="training"></a>

Train a Random Forest regression model optimized for CWSI prediction.

In [ ]:
# Model hyperparameters (optimized for regression)
MODEL_PARAMS = {
    'n_estimators': 200,           # Number of trees
    'max_depth': None,             # No depth limit for full learning
    'min_samples_split': 2,        # Minimum samples to split
    'min_samples_leaf': 1,         # Minimum samples per leaf
    'max_features': 'sqrt',        # Feature subset size
    'random_state': RANDOM_STATE,  # Reproducibility
    'n_jobs': -1,                  # Use all CPU cores
    'bootstrap': True              # Bootstrap sampling
}

print("  Initializing Random Forest Regressor with parameters:")
for param, value in MODEL_PARAMS.items():
    print(f"   • {param}: {value}")

# Initialize and train the model
model = RandomForestRegressor(**MODEL_PARAMS)
print(f"\n Training model on {X_train.shape[0]} samples...")
model.fit(X_train, y_train)
print(" Model training completed")

  Initializing Random Forest Regressor with parameters:
   • n_estimators: 200
   • max_depth: None
   • min_samples_split: 2
   • min_samples_leaf: 1
   • max_features: sqrt
   • random_state: 42
   • n_jobs: -1
   • bootstrap: True

 Training model on 1600 samples...
 Model training completed


## 5. Model Evaluation <a name="evaluation"></a>

Evaluate the regression model's performance using comprehensive metrics.

In [ ]:
# Generate predictions
y_pred = model.predict(X_test)

# Calculate regression metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Calculate additional metrics
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100  # Mean Absolute Percentage Error
explained_variance = 1 - (np.var(y_test - y_pred) / np.var(y_test))

print(" Model Performance Metrics:")
print("=" * 50)
print(f"Mean Absolute Error (MAE):     {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Percentage Error: {mape:.2f}%")
print(f"R² Score:                      {r2:.4f}")
print(f"Explained Variance:            {explained_variance:.4f}")

# Performance interpretation
print(f"\n Performance Rating:")
if r2 > 0.9:
    print("   Excellent (R² > 0.9)")
elif r2 > 0.8:
    print("   Very Good (R² > 0.8)")
elif r2 > 0.7:
    print("   Good (R² > 0.7)")
elif r2 > 0.6:
    print("   Fair (R² > 0.6)")
else:
    print("   Needs Improvement (R² ≤ 0.6)")

 Model Performance Metrics:
Mean Absolute Error (MAE):     0.0778
Root Mean Squared Error (RMSE): 0.0975
Mean Absolute Percentage Error: 82.34%
R² Score:                      0.8498
Explained Variance:            0.8499

 Performance Rating:
   Very Good (R² > 0.8)


In [ ]:
# Cross-validation for robust evaluation
CV_FOLDS = 5
print(f" Performing {CV_FOLDS}-fold cross-validation...")

cv_scores = cross_val_score(
    model, X, y,
    cv=CV_FOLDS,
    scoring='r2',
    n_jobs=-1
)

print(f"\n Cross-Validation Results:")
print(f"   R² scores: {cv_scores}")
print(f"   Mean R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"   Min R²: {cv_scores.min():.4f}")
print(f"   Max R²: {cv_scores.max():.4f}")

 Performing 5-fold cross-validation...

 Cross-Validation Results:
   R² scores: [0.8484626  0.85145207 0.85352529 0.82866058 0.82760035]
   Mean R²: 0.8419 ± 0.0114
   Min R²: 0.8276
   Max R²: 0.8535


In [ ]:
# Visualization: Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='black', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
            'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual CWSI')
axes[0].set_ylabel('Predicted CWSI')
axes[0].set_title(f'Predicted vs Actual CWSI\nR² = {r2:.4f}', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.6, color='darkorange', edgecolors='black', linewidth=0.5)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted CWSI')
axes[1].set_ylabel('Residuals (Actual - Predicted)')
axes[1].set_title(f'Residual Plot\nMean Residual = {residuals.mean():.4f}', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(" Top 10 Most Important Features:")
print("=" * 50)
for i, (_, row) in enumerate(feature_importance.head(10).iterrows(), 1):
    print(f"{i:2d}. {row['feature']:<20} {row['importance']:.4f}")

# Visualize feature importance
plt.figure(figsize=(12, 8))
bars = plt.barh(feature_importance['feature'][:15], feature_importance['importance'][:15])
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('Top 15 Feature Importances', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()  # Most important at top
plt.grid(True, alpha=0.3)

# Add value labels
for bar, importance in zip(bars, feature_importance['importance'][:15]):
    plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2, 
             f'{importance:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

 Top 10 Most Important Features:
 1. ndvi                 0.1957
 2. savi                 0.1915
 3. evi                  0.1769
 4. rh_percent           0.1388
 5. soil_moisture        0.0551
 6. vpd_kpa              0.0423
 7. irrigation_event     0.0410
 8. lst_celsius          0.0318
 9. ta_celsius           0.0227
10. et0_mm_day           0.0223


## 6. Stress Classification Logic <a name="classification"></a>

Implement conditional logic to classify stress levels based on predicted CWSI percentages.

In [ ]:
def classify_stress_level(cwsi_percentage):
    """
    Classify crop water stress level based on CWSI percentage.
    
    Parameters:
    cwsi_percentage (float): CWSI value as percentage (0-100)
    
    Returns:
    str: Stress level classification
    """
    if cwsi_percentage <= 24.4:
        return 'low'
    elif cwsi_percentage <= 49.7:
        return 'mild'
    elif cwsi_percentage <= 60:
        return 'medium'
    elif cwsi_percentage <= 80:
        return 'high'
    else:  # cwsi_percentage > 80
        return 'extreme'

# Test the classification function
print("Testing Stress Classification Logic:")
print("=" * 50)

test_values = [0, 10, 25, 30, 50, 60, 75, 80, 90, 95, 100]
for cwsi_pct in test_values:
    stress_level = classify_stress_level(cwsi_pct)
    print(f"   CWSI {cwsi_pct:3.0f}% → {stress_level}")



Testing Stress Classification Logic:
   CWSI   0% → low
   CWSI  10% → low
   CWSI  25% → mild
   CWSI  30% → mild
   CWSI  50% → medium
   CWSI  60% → medium
   CWSI  75% → high
   CWSI  80% → high
   CWSI  90% → extreme
   CWSI  95% → extreme
   CWSI 100% → extreme


In [ ]:
# Apply classification to predictions
y_pred_percentage = y_pred * 100  # Convert to percentage
predicted_stress_levels = [classify_stress_level(pct) for pct in y_pred_percentage]

# Compare with actual stress levels
# y_test is a numpy array, so it has no .index. Use the same train/test split indices.
idx = np.arange(len(df_processed))
_, _, _, _, _, test_idx = train_test_split(
    X, y, idx,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)
actual_stress_levels = df_processed.iloc[test_idx]['stress_label'].values

# Calculate classification accuracy
correct_predictions = sum(1 for pred, actual in zip(predicted_stress_levels, actual_stress_levels) if pred == actual)
classification_accuracy = correct_predictions / len(predicted_stress_levels) * 100

print(f" Stress Level Classification Results:")
print(f"   • Total predictions: {len(predicted_stress_levels)}")
print(f"   • Correct classifications: {correct_predictions}")
print(f"   • Classification accuracy: {classification_accuracy:.2f}%")

# Show classification distribution
pred_counts = pd.Series(predicted_stress_levels).value_counts().sort_index()
actual_counts = pd.Series(actual_stress_levels).value_counts().sort_index()

print(f"\n Predicted vs Actual Distribution:")
print(f"{'Level':<8} {'Predicted':>10} {'Actual':>10}")
print("-" * 30)
for level in ['low', 'mild', 'medium', 'high', 'extreme']:
    pred_count = pred_counts.get(level, 0)
    actual_count = actual_counts.get(level, 0)
    print(f"{level:<8} {pred_count:>10} {actual_count:>10}")

 Stress Level Classification Results:
   • Total predictions: 400
   • Correct classifications: 272
   • Classification accuracy: 68.00%

 Predicted vs Actual Distribution:
Level     Predicted     Actual
------------------------------
low             105        111
mild            132        128
medium           47        119
high             82         22
extreme          34         20


## 7. Model Persistence <a name="persistence"></a>

Save the trained model and preprocessing components for future use.

In [ ]:
# Save model and preprocessing objects
MODEL_FILENAME = 'cwsi_regression_model.pkl'
ENCODERS_FILENAME = 'categorical_encoders.pkl'

# Save the trained model
joblib.dump(model, MODEL_FILENAME)
print(f" Model saved as: {MODEL_FILENAME}")

# Save label encoders for categorical variables
joblib.dump(label_encoders, ENCODERS_FILENAME)
print(f" Encoders saved as: {ENCODERS_FILENAME}")

# Save feature information
model_info = {
    'features': FEATURE_COLUMNS,
    'categorical_columns': categorical_columns,
    'numerical_columns': numerical_columns,
    'model_params': MODEL_PARAMS,
    'training_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'performance_metrics': {
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
        'mape': mape,
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std()
    }
}

joblib.dump(model_info, 'model_metadata.pkl')
print(f" Model metadata saved as: model_metadata.pkl")

import os

print(f"\n Files saved in current directory:")
print(f"   • {MODEL_FILENAME} ({os.path.getsize(MODEL_FILENAME) / 1024:.1f} KB)")
print(f"   • {ENCODERS_FILENAME} ({os.path.getsize(ENCODERS_FILENAME) / 1024:.1f} KB)")
print(f"   • model_metadata.pkl ({os.path.getsize('model_metadata.pkl') / 1024:.1f} KB)")

 Model saved as: cwsi_regression_model.pkl
 Encoders saved as: categorical_encoders.pkl
 Model metadata saved as: model_metadata.pkl

 Files saved in current directory:
   • cwsi_regression_model.pkl (28484.5 KB)
   • categorical_encoders.pkl (1.1 KB)
   • model_metadata.pkl (0.8 KB)


## 8. Inference Example <a name="inference"></a>

Demonstrate how to use the trained model for new predictions.

In [ ]:
def predict_cwsi_stress(sample_data):
    """
    Predict CWSI percentage and stress level for new sample data.
    
    Parameters:
    sample_data (dict): Dictionary containing feature values
    
    Returns:
    tuple: (cwsi_percentage, stress_level)
    """
    # Load saved model and encoders
    loaded_model = joblib.load(MODEL_FILENAME)
    loaded_encoders = joblib.load(ENCODERS_FILENAME)
    loaded_info = joblib.load('model_metadata.pkl')
    
    # Prepare input data
    input_data = sample_data.copy()
    
    # Encode categorical variables
    for col in categorical_columns:
        if col in input_data:
            encoder = loaded_encoders[col]
            input_data[col] = encoder.transform([input_data[col]])[0]
    
    # Use the features the model was trained on
    required_features = loaded_info['features']
    
    # Ensure all required features are present
    missing = [f for f in required_features if f not in input_data]
    if missing:
        raise KeyError(f"Missing input feature(s): {missing}")

    # Create feature vector in correct order
    feature_vector = np.array([[input_data[feature] for feature in required_features]])
    
    # Make prediction
    cwsi_prediction = loaded_model.predict(feature_vector)[0]
    cwsi_percentage = cwsi_prediction * 100
    
    # Classify stress level
    stress_level = classify_stress_level(cwsi_percentage)

    # Return the output
    return cwsi_percentage, stress_level

# Example prediction
sample_field = {
    'region': 'Kairouan',
    'latitude': 35.67,
    'longitude': 10.10,
    'soil_type': 'clay-loam',
    'crop_type': 'Wheat',
    'month': 7,
    'lst_celsius': 38.5,
    'ndvi': 0.32,
    'savi': 0.28,
    'evi': 0.25,
    'ta_celsius': 34.0,
    'rh_percent': 28.0,
    'wind_ms': 3.2,
    'solar_wm2': 820.0,
    'vpd_kpa': 3.8,
    'et0_mm_day': 7.5,
    'soil_moisture': 0.11,
    'field_capacity': 0.28,
    'wilting_point': 0.10,
    'irrigation_event': 0
}

# Make prediction
predicted_cwsi_pct, predicted_stress = predict_cwsi_stress(sample_field)

print(" Field Stress Prediction Example:")
print("=" * 50)
print(f"Location: {sample_field['region']} - {sample_field['crop_type']} field")
print(f"Conditions: {sample_field['ta_celsius']}°C air temp, {sample_field['rh_percent']}% humidity")
print(f"Vegetation: NDVI = {sample_field['ndvi']}, LST = {sample_field['lst_celsius']}°C")
print(f"\n Predicted CWSI: {predicted_cwsi_pct:.1f}%")
print(f" Stress Level: {predicted_stress.upper()}")

# Provide irrigation recommendation based on stress level
recommendations = {
    'low': 'No irrigation needed - optimal water conditions',
    'mild': 'Monitor closely - consider light irrigation if conditions persist',
    'medium': 'Irrigation recommended within 2-3 days',
    'high': 'Immediate irrigation required',
    'extreme': 'Critical - immediate irrigation and field inspection needed'
}

print(f" Recommendation: {recommendations[predicted_stress]}")

 Field Stress Prediction Example:
Location: Kairouan - Wheat field
Conditions: 34.0°C air temp, 28.0% humidity
Vegetation: NDVI = 0.32, LST = 38.5°C

 Predicted CWSI: 48.3%
 Stress Level: MILD
 Recommendation: Monitor closely - consider light irrigation if conditions persist


---

##  Summary

This notebook implements a professional CWSI prediction system with the following features:

###  **Model Performance**
- **R² Score**: {r2:.4f}
- **MAE**: {mae:.4f}
- **RMSE**: {rmse:.4f}
- **Stress Classification Accuracy**: {classification_accuracy:.1f}%

###  **Key Features**
- Single regression model for CWSI prediction
- Automatic percentage conversion (0-100%)
- Conditional stress level classification
- Professional evaluation metrics
- Model persistence for deployment
- Feature importance analysis

###  **Stress Level Thresholds**
- **Low**: 0-24.4% (Healthy conditions)
- **Mild**: 24.4-49.7% (Early stress signs)
- **Medium**: 49.7-60% (Moderate stress)
- **High**: 60-80% (Severe stress)
- **Extreme**: 80-100% (Critical stress)

###  **Saved Files**
- `cwsi_regression_model.pkl` - Trained Random Forest model
- `categorical_encoders.pkl` - Label encoders for categorical features
- `model_metadata.pkl` - Model information and performance metrics

